# GPU risk tests, run 2: fix vLLM on a T4 (tests 3 and 4)

Run 1 passed tests 1–2. vLLM did not start: FlashInfer compiles a small kernel for Qwen3.5's norm layer on first use,
and the link step failed with `/usr/bin/ld: cannot find -lcuda`. Kaggle has the GPU driver (`libcuda.so.1`) but no
`libcuda.so` file where the linker looks.

This notebook tries fixes in a set order and stops at the first one that serves. It never starts an attempt after the
**60-minute fix limit**:

| Attempt | Change |
|---|---|
| A | Point the linker at the driver (`libcuda.so` link plus `LIBRARY_PATH`), same flags as run 1 |
| B | A + `--enforce-eager` (no CUDA graphs) |
| C | A + FlashInfer removed from the vLLM environment, so vLLM uses its own kernels |
| D | A + an older vLLM (0.17.x) |

If none works, test 3 takes its fallback. Every attempt's full log is saved in `vllm-logs/`, and its root-cause lines
go into the report.

**Before running (Kaggle):** Settings → Accelerator **GPU T4 x2** (one T4 is enough), Internet **On**;
Add-ons → Secrets → tick `HF_TOKEN`.
**For test 4 (recommended):** Add Input → Your Work → choose the run 1 notebook → Add. That attaches the adapter
trained in run 1. Without it, the notebook makes a random-weight adapter, which only proves vLLM can load one.
Then **Save Version → Save & Run All** (about 20–60 minutes) and copy everything from `===== VLLM FIX REPORT =====`
to the end.

In [ ]:
# 0. Setup: helpers, Hugging Face login, the adapter from run 1
import glob, json, os, re, signal, subprocess, sys, time, traceback, urllib.request

os.makedirs("vllm-logs", exist_ok=True)
VLLM_GPU = "0"                       # nothing else uses the GPU in this notebook
REPORT, ATTEMPTS, NOTES = [], [], []

def record(test, ok, detail, fallback=""):
    REPORT.append({"test": test, "result": "PASS" if ok else "FAIL",
                   "detail": str(detail)[-2500:], "fallback": "" if ok else fallback})
    print(f"\n### {test}: {'PASS' if ok else 'FAIL'}\n{detail}")

def err():
    return traceback.format_exc()[-1500:]

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = ""
if HF_TOKEN:                        # the base model is public; the token only avoids download rate limits
    os.environ["HF_TOKEN"] = HF_TOKEN

SYSTEM = "You are Nova, NimbusCart's support assistant."
TOOLS = [{"type": "function", "function": {
    "name": "get_order_status",
    "description": "Status of one of the signed-in customer's orders",
    "parameters": {"type": "object", "properties": {"order_id": {"type": "integer"}}, "required": ["order_id"]}}}]

found = sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True))
ADAPTER_DIR = os.path.dirname(found[0]) if found else ""
ADAPTER_KIND = "run 1 adapter (trained 50 steps)" if found else ""
print(sh("nvidia-smi -L"))
print("adapter from run 1:", ADAPTER_DIR or "not attached (a random-weight adapter will be made for test 4)")

In [ ]:
# 1. Install vLLM in its own environment (same version as run 1) and the OpenAI client here
%pip install -q uv openai
!uv venv -q /tmp/vllm-env --python 3.12
!uv pip install -q --python /tmp/vllm-env/bin/python "vllm==0.34.0"

def venv_versions(py):
    code = ("import importlib.metadata as m, json; d = {}\n"
            "for n in ['vllm', 'torch', 'transformers', 'triton', 'flashinfer-python']:\n"
            "    try: d[n] = m.version(n)\n"
            "    except Exception: d[n] = 'not installed'\n"
            "print(json.dumps(d))")
    r = subprocess.run([py, "-c", code], capture_output=True, text=True)
    return json.loads(r.stdout or "{}")

print(venv_versions("/tmp/vllm-env/bin/python"))

In [ ]:
# 2. Diagnose and fix the linker problem: give the linker a libcuda.so that points at Kaggle's driver
ENV_INFO = {
    "driver": sh("nvidia-smi --query-gpu=name,driver_version,compute_cap --format=csv,noheader"),
    "system nvcc": sh("nvcc --version | tail -2") or "not found",
    "/usr/local/cuda/lib64/stubs": sh("ls /usr/local/cuda/lib64/stubs 2>&1 | head -5"),
    "ldconfig libcuda": sh("ldconfig -p | grep -E 'libcuda\\.so' | head -5"),
}
candidates = []
for pattern in ["/usr/local/cuda*/**/stubs/libcuda.so", "/usr/lib/x86_64-linux-gnu/libcuda.so*",
                "/usr/local/nvidia/lib64/libcuda.so*", "/usr/lib64/libcuda.so*"]:
    candidates += glob.glob(pattern, recursive=True)
candidates += re.findall(r"=>\s*(\S*libcuda\.so[.\d]*)", ENV_INFO["ldconfig libcuda"])
candidates = [c for c in dict.fromkeys(candidates) if os.path.exists(c)]
LIBCUDA = candidates[0] if candidates else ""

LINK_DIR = "/tmp/cuda-link"
os.makedirs(LINK_DIR, exist_ok=True)
if LIBCUDA:
    target = os.path.join(LINK_DIR, "libcuda.so")
    if os.path.lexists(target):
        os.remove(target)
    os.symlink(os.path.realpath(LIBCUDA), target)
LINK_ENV = {"LIBRARY_PATH": LINK_DIR + ":" + os.environ.get("LIBRARY_PATH", "")}
ENV_INFO["libcuda used for linking"] = LIBCUDA or "NONE FOUND"
for k, v in ENV_INFO.items():
    print(f"{k}: {v}")

In [ ]:
# 3. Start helpers: launch vLLM, wait for it, pull root-cause lines out of a failed log
BASE_FLAGS = ["--dtype", "half", "--max-model-len", "8192", "--language-model-only",
              "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder", "--port", "8000"]
LORA_FLAGS = lambda path: ["--enable-lora", "--lora-modules", f"nova-test={path}", "--max-lora-rank", "16"]

PRIORITY = ("cannot find", "No such file", "out of memory", "CUDA error", "undefined symbol",
            "ImportError", "ModuleNotFoundError", "not supported", "Error:", "error:")
def root_cause(text):
    hits = []
    for line in text.splitlines():
        s = re.sub(r"^\(\w+ pid=\d+\)\s+\w+\s+\S+\s+\S+\s+\[[^\]]*\]\s?", "", line).strip()
        if not s or s.startswith("File ") or set(s) <= set("^~ "):
            continue
        for rank, key in enumerate(PRIORITY):
            if key in s:
                hits.append((rank, s[:300]))
                break
    seen, lines_ = set(), []
    for _, s in sorted(hits, key=lambda h: h[0]):
        if s not in seen:
            seen.add(s); lines_.append(s)
    return "\n".join(lines_[:10]) or text[-1500:]

def stop(proc):
    if proc and proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGTERM); proc.wait(60)
        except Exception:
            os.killpg(proc.pid, signal.SIGKILL)
    time.sleep(5)

def serve(name, vllm_bin, extra=(), lora=None, wait_s=900):
    cmd = [vllm_bin, "serve", "Qwen/Qwen3.5-2B", *BASE_FLAGS, *extra] + (LORA_FLAGS(lora) if lora else [])
    log_path = f"vllm-logs/{re.sub(r'[^a-z0-9]+', '-', name.lower()).strip('-')}.log"
    log = open(log_path, "w")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=VLLM_GPU, **LINK_ENV)
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    t0 = time.time()
    while time.time() - t0 < wait_s:
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
            secs = time.time() - t0
            print(f"[{name}] vLLM ready in {secs:.0f}s")
            return proc, secs, ""
        except Exception:
            time.sleep(10)
    secs = time.time() - t0
    stop(proc)
    cause = root_cause(open(log_path).read())
    print(f"[{name}] did not start after {secs:.0f}s. Root cause lines:\n{cause}")
    return None, secs, cause

In [ ]:
# 4. TEST 3: try the fixes in order (60-minute limit), then ask for one parsed tool call
FALLBACK_3 = "Run evals with Transformers or llama.cpp; serve on an L4- or A10-class GPU"
FALLBACK_4 = "Merge the adapter into the base model instead of serving it separately"
FIX_LIMIT_S = 60 * 60

def setup_no_flashinfer():
    py = "/tmp/vllm-env/bin/python"
    names = [n for n, v in venv_versions(py).items() if n.startswith("flashinfer") and v != "not installed"]
    extra = sh(f"uv pip list --python {py} 2>/dev/null | awk '{{print $1}}' | grep -i '^flashinfer'").split()
    pkgs = sorted(set(names + extra))
    if pkgs:
        print(sh(f"uv pip uninstall --python {py} " + " ".join(pkgs)))
    return "/tmp/vllm-env/bin/vllm", py

def setup_old_vllm():
    print(sh("uv venv -q /tmp/vllm-old --python 3.12 && "
             "uv pip install -q --python /tmp/vllm-old/bin/python 'vllm>=0.17,<0.18'"))
    return "/tmp/vllm-old/bin/vllm", "/tmp/vllm-old/bin/python"

PLAN = [
    ("A: libcuda link fix", lambda: ("/tmp/vllm-env/bin/vllm", "/tmp/vllm-env/bin/python"), []),
    ("B: A + enforce-eager", lambda: ("/tmp/vllm-env/bin/vllm", "/tmp/vllm-env/bin/python"), ["--enforce-eager"]),
    ("C: A + no FlashInfer", setup_no_flashinfer, []),
    ("D: A + vLLM 0.17.x", setup_old_vllm, []),
]

server, WORKING, SERVE_VERSIONS = None, None, {}
fix_start = time.time()
for name, setup, extra in PLAN:
    used = time.time() - fix_start
    if used > FIX_LIMIT_S:
        ATTEMPTS.append({"attempt": name, "result": "not run (60-minute limit reached)", "seconds": 0, "cause": ""})
        continue
    try:
        vllm_bin, py = setup()
    except Exception:
        ATTEMPTS.append({"attempt": name, "result": "setup failed", "seconds": 0, "cause": err()[-600:]})
        continue
    proc, secs, cause = serve(name, vllm_bin, extra)
    ATTEMPTS.append({"attempt": name, "result": "started" if proc else "did not start",
                     "seconds": round(secs), "cause": cause})
    if proc:
        server, WORKING = proc, {"name": name, "vllm_bin": vllm_bin, "py": py, "extra": extra}
        SERVE_VERSIONS = venv_versions(py)
        break
FIX_MINUTES = round((time.time() - fix_start) / 60)

if server is None:
    record("3. vLLM parsed tool call", False,
           f"No attempt started vLLM within the 60-minute limit ({FIX_MINUTES} min used). See the attempts table.",
           FALLBACK_3)
else:
    from openai import OpenAI
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")
    try:
        t0 = time.time()
        r = client.chat.completions.create(model="Qwen/Qwen3.5-2B", tools=TOOLS, temperature=0,
            messages=[{"role": "user", "content": "Where is my order 4417?"}])
        calls = r.choices[0].message.tool_calls
        ok = bool(calls) and calls[0].function.name == "get_order_status" \
            and json.loads(calls[0].function.arguments).get("order_id") in (4417, "4417")
        record("3. vLLM parsed tool call", ok,
               f"started with: {WORKING['name']} | {time.time() - t0:.2f}s for the call\n"
               f"tool_calls: {calls}\ncontent: {r.choices[0].message.content!r}", FALLBACK_3)
    except Exception:
        record("3. vLLM parsed tool call", False, err(), FALLBACK_3)

In [ ]:
# 5. Sampling comparison through vLLM (repeats the sampling experiment in notes/generation-experiments/, now with presence_penalty) and a speed check
SAMPLING, SPEED = {}, ""
if server is not None:
    try:
        ask = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Hi, where is my order 4417?"}]
        t0 = time.time()
        c = client.chat.completions.create(model="Qwen/Qwen3.5-2B", messages=ask, temperature=0, max_tokens=300)
        n = c.usage.completion_tokens
        SPEED = f"{n} tokens in {time.time() - t0:.1f}s ({n / (time.time() - t0):.1f} tok/s), greedy, one request"
        print(SPEED)
        for name, kw in {"temperature 0.2": dict(temperature=0.2),
                         "model card (temp 1.0, top_p 1.0, top_k 20, presence_penalty 2.0)":
                             dict(temperature=1.0, top_p=1.0, presence_penalty=2.0, extra_body={"top_k": 20})}.items():
            runs = []
            for seed in range(5):
                c = client.chat.completions.create(model="Qwen/Qwen3.5-2B", messages=ask, max_tokens=600,
                                                   seed=seed, **kw)
                runs.append({"tokens": c.usage.completion_tokens, "finish": c.choices[0].finish_reason,
                             "text": (c.choices[0].message.content or "")[:160]})
            SAMPLING[name] = runs
            print(f"\n{name}: lengths {[x['tokens'] for x in runs]}")
            for x in runs:
                print(f"  ({x['tokens']} tokens, {x['finish']}) {x['text']}")
    except Exception:
        print("sampling comparison failed:", err())
    stop(server)
    server = None

In [ ]:
# 6. TEST 4: serve an adapter as a LoRA module next to the base model
RANDOM_ADAPTER = r'''
import sys, torch, transformers
from peft import LoraConfig, get_peft_model
Auto = getattr(transformers, "AutoModelForMultimodalLM", None) or transformers.AutoModelForImageTextToText
m = Auto.from_pretrained("Qwen/Qwen3.5-2B", dtype=torch.float16, device_map="cpu")
cfg = LoraConfig(r=16, lora_alpha=16, init_lora_weights=False,
                 target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)")
get_peft_model(m, cfg).save_pretrained(sys.argv[1])
print("saved random adapter to", sys.argv[1])
'''

if WORKING is None:
    record("4. vLLM serves LoRA adapter", False, "Not tried: vLLM did not start (test 3).", FALLBACK_4)
else:
    py = WORKING["py"]
    if not ADAPTER_DIR:
        try:
            print(sh(f"uv pip install -q --python {py} peft"))
            open("/tmp/random_adapter.py", "w").write(RANDOM_ADAPTER)
            print(sh(f"{py} /tmp/random_adapter.py /kaggle/working/random-adapter"))
            if os.path.exists("/kaggle/working/random-adapter/adapter_config.json"):
                ADAPTER_DIR, ADAPTER_KIND = "/kaggle/working/random-adapter", "random-weight adapter (mechanism only)"
        except Exception:
            print("could not make a random adapter:", err())
    if not ADAPTER_DIR:
        record("4. vLLM serves LoRA adapter", False, "No adapter: run 1's output is not attached and "
               "making a random one failed.", FALLBACK_4)
    else:
        tries = [WORKING["extra"]] + ([] if "--enforce-eager" in WORKING["extra"] else [WORKING["extra"] + ["--enforce-eager"]])
        lora_server, lora_causes = None, []
        for extra in tries:
            label = f"LoRA: {WORKING['name']}" + (" + enforce-eager" if extra is not WORKING["extra"] else "")
            lora_server, secs, cause = serve(label, WORKING["vllm_bin"], extra, lora=ADAPTER_DIR)
            ATTEMPTS.append({"attempt": label, "result": "started" if lora_server else "did not start",
                             "seconds": round(secs), "cause": cause})
            if lora_server:
                break
            lora_causes.append(f"{label}:\n{cause}")
        if lora_server is None:
            record("4. vLLM serves LoRA adapter", False, "\n\n".join(lora_causes), FALLBACK_4)
        else:
            try:
                from openai import OpenAI
                client = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")
                ask = [{"role": "user", "content": "I want to cancel my order"}]
                b = client.chat.completions.create(model="Qwen/Qwen3.5-2B", messages=ask, temperature=0,
                                                   max_tokens=150).choices[0].message.content or ""
                l = client.chat.completions.create(model="nova-test", messages=ask, temperature=0,
                                                   max_tokens=150).choices[0].message.content or ""
                record("4. vLLM serves LoRA adapter", bool(l.strip()) and l.strip() != b.strip(),
                       f"adapter: {ADAPTER_KIND}\n--- base ---\n{b}\n--- nova-test ---\n{l}", FALLBACK_4)
            except Exception:
                record("4. vLLM serves LoRA adapter", False, err(), FALLBACK_4)
            stop(lora_server)

In [ ]:
# 7. REPORT: copy everything from the next line to the end and paste it back
print("===== VLLM FIX REPORT =====")
print("\n--- environment ---")
for k, v in ENV_INFO.items():
    print(f"{k}: {v}")
print(f"\n--- fix attempts ({FIX_MINUTES} min used of the 60-minute limit) ---")
print("| Attempt | Result | Seconds |\n|---|---|---|")
for a in ATTEMPTS:
    print(f"| {a['attempt']} | {a['result']} | {a['seconds']} |")
for a in ATTEMPTS:
    if a["cause"]:
        print(f"\n[{a['attempt']}] root cause lines:\n{a['cause']}")
print("\n| Test | Result | Fallback if failed |\n|---|---|---|")
for r in REPORT:
    print(f"| {r['test']} | {r['result']} | {r['fallback']} |")
print("\n--- serving environment that worked ---")
if WORKING:
    print("attempt:", WORKING["name"], "| extra flags:", " ".join(WORKING["extra"]) or "none")
    for k, v in SERVE_VERSIONS.items():
        print(f"{k}=={v}")
else:
    print("none")
print("\n--- details ---")
for r in REPORT:
    print(f"\n## {r['test']}: {r['result']}\n{r['detail']}")
if SPEED:
    print("\nspeed:", SPEED)
if SAMPLING:
    print("\n--- sampling lengths (vLLM) ---")
    for name, runs in SAMPLING.items():
        print(f"{name}: {[(x['tokens'], x['finish']) for x in runs]}")
print("===== END OF REPORT =====")